# Data Analysis for BSSE Project

This notebook contains Ryan's analysis of Ibrahim's data. 

## Reading in Data

The goal of this section is to read the data into data structures that are conducive to analysis. At the same time, we want to build in sanity checks to help us ensure that calculations run successfully and that calculations are what we think they are. The end goal is to have the following data structures:

- Map from geometries to "short names"
- Table of energies with one row per "short name"

To define the short names:

- `{system}_{parameters}` where `{system}` is one or more underscore-separated monomers, e.g., `Ne_Ne_{parameters}` is a neon dimer.
-  Monomers in `()` are ghosted, i.e., `Ne_(Ne)_{parameters}` computes the energy of the first Ne atom using the dimer basis set.

Misc. Notes.

- Data for the project lives in `bsse_db/data/monomer_name`, where "monomer_name" is the molecular formula of the monomers in the cluster.
- We are going to assume that for a monomer containing $n$ atoms, the first $n$ atoms in a file belong to monomer 1, the next $n$ belong to monomer 2, the next $n$ belong to monomer 3, etc.
- Ghost atoms in NWChem are specified by prepending `bq` to the atomic symbol.
- Initial data allowed the molecular system to be reoriented. This leads to small geometric differences between the supersystem and subsystem geometries. 


In [10]:
import tarfile
import itertools
import os
import math
import pandas as pd
from nwchem_helpers.parse_nwchem_output import parse_nwchem_output

### Ne Clusters

- `Ne_dimers.tar` contain CCSD(T)/aug-cc-pvdz calculations (TODO: verify).
- `Ne_Ne_distance_x_y_z` directory contains a dimer where one of the monomers has been translated by $\vec{r} = (x,y,z)^T$.
- Translating like this duplicates effort because of the system's symmetry (i.e., only the distance matters)
- No 'output_E_B_B.txt' because monomers are the same.

In [17]:
nsteps    = 7    # The total number of displacements along each axis
step_size = 0.25 # How much we displace for each step.

ne_dimer_distances = set()
geometries = {}
energies = {}

def parse_tarred_file(tarball, file_path):
    f = tarball.extractfile(file_path)
    content = f.read().decode("utf-8").split('\n')
    return parse_nwchem_output(iter(content))

def save_state(key, results, geometries, energies):
    geom = results.pop('Input Geometry (angstroms)')
    if key in geometries:
        assert geometries[key] == geom
    else:
        geometries[r] = geom

    if key in energies:
        for egy_type, egy_value in energies[key].items():
            # SCF convergence is 1E-6 so can't expect better
            assert math.isclose(float(egy_value), float(results[egy_type]), abs_tol=1E-6)
    else:
        energies[key] = results

def to_point(atom):
    return [float(atom[i]) for i in range(1, 4)]

def compute_ne_distance(geom):
    carts = [to_point(geom[i]) for i in range(2)]
    return math.dist(carts[0], carts[1])

with tarfile.open('data/Ne/Ne_dimers.tar', 'r') as tarball:
    all_names = tarball.getnames() # Gets all the directories and files inside the tarball
    
    for dx, dy, dz in itertools.product(range(1, nsteps), range(1, nsteps), range(1, nsteps)):
        x, y, z = (dx * step_size, dy * step_size, dz * step_size)
        directory_name = os.path.join('Ne_dimers', 'Ne_Ne_distance_{}_{}_{}'.format(x, y, z))
    
        if directory_name in all_names: # Checks translation is in tarball
            
            # Extract the results from the dimer file
            file_name = os.path.join(directory_name, 'output_E_AB_AB.txt')
            results = parse_tarred_file(tarball, file_name)

            # Compute and record the separation distance
            geom = results['Input Geometry (angstroms)']
            r = compute_ne_distance(geom)
            ne_dimer_distances.add(r)

            # Save the state
            key = 'Ne_Ne_{}'.format(r)
            save_state(key, results, geometries, energies)

            # Extract the results for monomer 0 in the dimer basis
            file_name = os.path.join(directory_name, 'output_E_AB_A.txt')
            results = parse_tarred_file(tarball, file_name)

            # Sanity check it's the same distance
            geom = results['Input Geometry (angstroms)']
            r_new = compute_ne_distance(geom)
            assert math.isclose(r, r_new, abs_tol=1E-7) # NWChem only prints about 8 decimal places

            # Save the state (use dimer distance for consistency)
            key = 'Ne_(Ne)_{}'.format(r)
            save_state(key, results, geometries, energies)

            # Extract the results for monomer 1 in the dimer basis
            file_name = os.path.join(directory_name, 'output_E_AB_B.txt')
            results = parse_tarred_file(tarball, file_name)

            # Sanity check it's the same distance
            geom = results['Input Geometry (angstroms)']
            r_new = compute_ne_distance(geom)
            assert math.isclose(r, r_new, abs_tol=1E-7) # NWChem only prints about 8 decimal places

            # Save the state (use dimer distance for consistency)
            key = '(Ne)_Ne_{}'.format(r)
            save_state(key, results, geometries, energies)

            # Extract the results for monomer 0 in the monomer basis
            file_name = os.path.join(directory_name, 'output_E_A_A.txt')
            results = parse_tarred_file(tarball, file_name)

            # Save the state (use dimer distance for consistency)
            key = 'Ne'
            save_state(key, results, geometries, energies)                

In [20]:
pd.DataFrame(energies).transpose()

,Total SCF Energy (a.u.),Total MP2 Energy (a.u.),Total CCSD Energy (a.u.),Total CCSD(T) Energy (a.u.)
Ne_Ne_1.5411035,-256.874113911105,-257.292842423520881,-257.299248528910653,-257.305108143114182
Ne_(Ne)_1.5411035,-128.496953904925,-128.706906455561153,-128.710024487065823,-128.712905546045221
(Ne)_Ne_1.5411035,-128.496953904925,-128.706906455561125,-128.710024474655171,-128.712905531504191
Ne,-128.496349730514,-128.705409599288487,-128.708487837634010,-128.711294152566950
Ne_Ne_1.60078106,-256.902970297102,-257.321600541022690,-257.328049289253840,-257.333887071280117
...,...,...,...,...
Ne_(Ne)_2.46221446,-128.496439816931,-128.705570759155108,-128.708655551024947,-128.711479578227227
(Ne)_Ne_2.46221446,-128.496439816931,-128.705570759155023,-128.708655541533858,-128.711479570102568
Ne_Ne_2.59807622,-256.991935285700,-257.410324413582885,-257.416530286745228,-257.422199193337235
Ne_(Ne)_2.59807622,-128.496428160688,-128.705549214647675,-128.708635670441282,-128.711457349988592
